# Veri Seti Analizi / Dataset Analysis

Bu notebook, datasets/defect_detection veri setini analiz eder ve görselleştirir.
This notebook analyzes and visualizes the defect_detection dataset.

## Dataset Structure:
```
datasets/defect_detection/
├── dataset.yaml          # YOLO format config
├── train/
│   ├── images/          # Training images
│   └── labels/         # Training labels (YOLO format)
├── val/
│   ├── images/          # Validation images
│   └── labels/         # Validation labels
└── test/
    ├── images/          # Test images
    └── labels/         # Test labels
```

## YOLO Label Format:
`class_id x_center y_center width height`
- Coordinates are normalized (0-1)
- Bounding box format: xywh (center format)

In [ ]:
# Import required libraries
import os
import sys
from pathlib import Path

# Add project root to path
project_root = Path('../').resolve()
sys.path.insert(0, str(project_root))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import yaml
import random

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("Libraries imported successfully!")

## Veri Seti Yükleme / Loading Dataset Configuration

In [ ]:
# Dataset paths
DATASET_ROOT = Path("../datasets/defect_detection")
DATASET_YAML = DATASET_ROOT / "dataset.yaml"

# Load dataset configuration
with open(DATASET_YAML, 'r', encoding='utf-8') as f:
    dataset_config = yaml.safe_load(f)

print("=" * 50)
print("VERİ SETİ YAPILANDIRMASI / DATASET CONFIGURATION")
print("=" * 50)
print(f"Dataset Name: {dataset_config.get('name', 'N/A')}")
print(f"Version: {dataset_config.get('version', 'N/A')}")
print(f"Description: {dataset_config.get('description', 'N/A')}")
print(f"Number of Classes: {dataset_config.get('nc', 'N/A')}")
print(f"Class Names: {dataset_config.get('names', {})}")
print(f"Default Image Size: {dataset_config.get('imgsz', 'N/A')}")

# Extract class names
class_names = dataset_config.get('names', {})
print(f"\nClasses: {class_names}")

## Klasör Yapısını İnceleme / Exploring Folder Structure

In [ ]:
def explore_folder_structure(root_path, max_depth=3, indent=0):
    """
    Recursively explore and display folder structure.
    """
    root = Path(root_path)
    if not root.exists():
        print(f"Path does not exist: {root_path}")
        return
    
    prefix = "  " * indent
    if root.is_file():
        size = os.path.getsize(root) / 1024  # KB
        print(f"{prefix}{root.name} ({size:.1f} KB)")
        return
    
    print(f"{prefix}{root.name}/")
    
    if indent < max_depth:
        items = sorted(root.iterdir(), key=lambda x: (x.is_file(), x.name))
        for item in items[:20]:  # Limit items shown
            explore_folder_structure(item, max_depth, indent + 1)
        if len(items) > 20:
            print(f"{prefix}  ... and {len(items) - 20} more items")

# Explore dataset structure
print("DATASET FOLDER STRUCTURE:")
print("=" * 50)
explore_folder_structure(DATASET_ROOT)

## Veri Seti İstatistikleri / Dataset Statistics

In [ ]:
def count_images_and_labels(dataset_root, split):
    """
    Count images and labels in a dataset split.
    """
    images_dir = dataset_root / split / "images"
    labels_dir = dataset_root / split / "labels"
    
    # Count images
    if images_dir.exists():
        num_images = len(list(images_dir.glob("*.jpg"))) + \
                     len(list(images_dir.glob("*.jpeg"))) + \
                     len(list(images_dir.glob("*.png")))
    else:
        num_images = 0
    
    # Count labels
    if labels_dir.exists():
        num_labels = len(list(labels_dir.glob("*.txt")))
    else:
        num_labels = 0
    
    return num_images, num_labels

# Count for each split
splits = ['train', 'val', 'test']
stats = {'split': [], 'images': [], 'labels': [], 'annotations': []}

print("DATASET STATISTICS:")
print("=" * 50)

for split in splits:
    num_images, num_labels = count_images_and_labels(DATASET_ROOT, split)
    stats['split'].append(split)
    stats['images'].append(num_images)
    stats['labels'].append(num_labels)
    stats['annotations'].append(num_labels)  # Assuming 1 label file per image
    print(f"{split.upper():10s} | Images: {num_images:5d} | Labels: {num_labels:5d}")

print("-" * 50)
print(f"{'TOTAL':10s} | Images: {sum(stats['images']):5d} | Labels: {sum(stats['labels']):5d}")

# Create dataframe
df_stats = pd.DataFrame(stats)
print("\n", df_stats)

## Etiketleri Analiz Etme / Analyzing Labels

In [ ]:
def parse_yolo_labels(labels_dir, class_names):
    """
    Parse YOLO format labels and collect statistics.
    """
    if not labels_dir.exists():
        return None, {}
    
    all_boxes = []
    class_counts = Counter()
    
    label_files = list(labels_dir.glob("*.txt"))
    
    for label_file in label_files:
        with open(label_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_id = int(parts[0])
                    x_center = float(parts[1])
                    y_center = float(parts[2])
                    width = float(parts[3])
                    height = float(parts[4])
                    
                    class_counts[class_id] += 1
                    all_boxes.append({
                        'class_id': class_id,
                        'x_center': x_center,
                        'y_center': y_center,
                        'width': width,
                        'height': height,
                        'area': width * height
                    })
    
    return all_boxes, class_counts

# Analyze labels for each split
label_stats = {}

for split in splits:
    labels_dir = DATASET_ROOT / split / "labels"
    boxes, class_counts = parse_yolo_labels(labels_dir, class_names)
    
    if boxes:
        label_stats[split] = {
            'total_boxes': len(boxes),
            'class_counts': class_counts,
            'boxes': boxes
        }
    else:
        label_stats[split] = {
            'total_boxes': 0,
            'class_counts': Counter(),
            'boxes': []
        }

print("LABEL ANALYSIS:")
print("=" * 50)

for split in splits:
    stats = label_stats[split]
    print(f"\n{split.upper()}:")
    print(f"  Total bounding boxes: {stats['total_boxes']}")
    if stats['class_counts']:
        print("  Class distribution:")
        for class_id, count in sorted(stats['class_counts'].items()):
            class_name = class_names.get(class_id, f"class_{class_id}")
            print(f"    {class_name}: {count}")

## Görselleştirme - Sınıf Dağılımı / Visualization - Class Distribution

In [ ]:
# Aggregate class counts across all splits
total_class_counts = Counter()
for split in splits:
    total_class_counts.update(label_stats[split]['class_counts'])

# Create dataframe
class_df = pd.DataFrame([
    {'class_id': k, 'class_name': class_names.get(k, f"class_{k}"), 'count': v}
    for k, v in total_class_counts.items()
])

if not class_df.empty:
    class_df = class_df.sort_values('count', ascending=False)

# Plot class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
if not class_df.empty:
    ax1 = axes[0]
    bars = ax1.bar(class_df['class_name'], class_df['count'], color='steelblue', edgecolor='black')
    ax1.set_xlabel('Sınıf / Class')
    ax1.set_ylabel('Örnek Sayısı / Count')
    ax1.set_title('Sınıf Dağılımı / Class Distribution')
    ax1.tick_params(axis='x', rotation=45)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height)}', ha='center', va='bottom')

# Pie chart
if not class_df.empty:
    ax2 = axes[1]
    colors = plt.cm.Set3(np.linspace(0, 1, len(class_df)))
    wedges, texts, autotexts = ax2.pie(
        class_df['count'], 
        labels=class_df['class_name'],
        autopct='%1.1f%%',
        colors=colors,
        explode=[0.02] * len(class_df)
    )
    ax2.set_title('Sınıf Yüzdeleri / Class Percentages')

plt.tight_layout()
plt.savefig('../outputs/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nClass distribution plot saved to: ../outputs/class_distribution.png")

## Örnek Görüntüler / Sample Images

In [ ]:
def load_yolo_image_and_labels(image_path, label_path, img_size=640):
    """
    Load image and draw YOLO format bounding boxes.
    """
    # Load image
    image = cv2.imread(str(image_path))
    if image is None:
        return None, None
    
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image_rgb.shape[:2]
    
    # Load labels
    boxes = []
    if label_path and label_path.exists():
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_id = int(parts[0])
                    x_center = float(parts[1]) * w
                    y_center = float(parts[2]) * h
                    width = float(parts[3]) * w
                    height = float(parts[4]) * h
                    
                    # Convert to corner format
                    x1 = int(x_center - width / 2)
                    y1 = int(y_center - height / 2)
                    x2 = int(x_center + width / 2)
                    y2 = int(y_center + height / 2)
                    
                    boxes.append({
                        'class_id': class_id,
                        'class_name': class_names.get(class_id, f"class_{class_id}"),
                        'bbox': [x1, y1, x2, y2]
                    })
    
    return image_rgb, boxes

# Get sample images from training set
train_images_dir = DATASET_ROOT / "train" / "images"
train_labels_dir = DATASET_ROOT / "train" / "labels"

if train_images_dir.exists():
    image_files = sorted(list(train_images_dir.glob("*.jpg")) + 
                        list(train_images_dir.glob("*.png")))[:4]
    
    print(f"Found {len(image_files)} sample images")
    
    # Display sample images
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    axes = axes.flatten()
    
    colors = plt.cm.tab10(np.linspace(0, 1, 10))
    
    for idx, img_path in enumerate(image_files):
        label_path = train_labels_dir / (img_path.stem + '.txt')
        image, boxes = load_yolo_image_and_labels(img_path, label_path)
        
        if image is not None:
            # Draw boxes
            for box in boxes:
                x1, y1, x2, y2 = box['bbox']
                color = colors[box['class_id'] % len(colors)]
                cv2.rectangle(image, (x1, y1), (x2, y2), 
                            color=(int(color[0]*255), int(color[1]*255), int(color[2]*255)), 
                            thickness=2)
                cv2.putText(image, box['class_name'], (x1, y1-5),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                           color=(int(color[0]*255), int(color[1]*255), int(color[2]*255)),
                           thickness=2)
            
            axes[idx].imshow(image)
            axes[idx].set_title(f"{img_path.name}\n{len(boxes)} boxes")
            axes[idx].axis('off')
        else:
            axes[idx].text(0.5, 0.5, "Image not found", ha='center', va='center')
            axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print(f"Training images directory not found: {train_images_dir}")

## Görselleştirme - Bounding Box Boyutları / Visualization - Bounding Box Sizes

In [ ]:
# Analyze bounding box sizes
all_boxes = []
for split in splits:
    all_boxes.extend(label_stats[split]['boxes'])

if all_boxes:
    widths = [box['width'] for box in all_boxes]
    heights = [box['height'] for box in all_boxes]
    areas = [box['area'] for box in all_boxes]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Width distribution
    axes[0].hist(widths, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Genişlik (normalized) / Width')
    axes[0].set_ylabel('Frekans / Frequency')
    axes[0].set_title('Bounding Box Genişliği / Box Width')
    axes[0].axvline(np.mean(widths), color='red', linestyle='--', label=f'Mean: {np.mean(widths):.2f}')
    axes[0].legend()
    
    # Height distribution
    axes[1].hist(heights, bins=30, color='forestgreen', edgecolor='black', alpha=0.7)
    axes[1].set_xlabel('Yükseklik (normalized) / Height')
    axes[1].set_ylabel('Frekans / Frequency')
    axes[1].set_title('Bounding Box Yüksekliği / Box Height')
    axes[1].axvline(np.mean(heights), color='red', linestyle='--', label=f'Mean: {np.mean(heights):.2f}')
    axes[1].legend()
    
    # Area distribution
    axes[2].hist(areas, bins=30, color='coral', edgecolor='black', alpha=0.7)
    axes[2].set_xlabel('Alan (normalized) / Area')
    axes[2].set_ylabel('Frekans / Frequency')
    axes[2].set_title('Bounding Box Alanı / Box Area')
    axes[2].axvline(np.mean(areas), color='red', linestyle='--', label=f'Mean: {np.mean(areas):.2f}')
    axes[2].legend()
    
    plt.tight_layout()
    plt.savefig('../outputs/bbox_sizes.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nBounding Box Statistics:")
    print(f"  Mean width: {np.mean(widths):.3f}")
    print(f"  Mean height: {np.mean(heights):.3f}")
    print(f"  Mean area: {np.mean(areas):.3f}")
    print(f"  Min area: {np.min(areas):.3f}")
    print(f"  Max area: {np.max(areas):.3f}")
else:
    print("No bounding boxes found in dataset.")

## Veri Artırma Görselleştirme / Data Augmentation Visualization

In [ ]:
# Data augmentation techniques visualization
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Define augmentation pipeline
def get_augmentation_pipeline():
    """
    Get YOLO-compatible augmentation pipeline.
    """
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.RandomRotate90(p=0.3),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=15, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
        A.Blur(blur_limit=3, p=0.3),
    ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

# Load a sample image
if train_images_dir.exists() and image_files:
    sample_image_path = image_files[0]
    sample_label_path = train_labels_dir / (sample_image_path.stem + '.txt')
    
    # Load image
    image = cv2.imread(str(sample_image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]
    
    # Load bboxes
    bboxes = []
    class_labels = []
    if sample_label_path.exists():
        with open(sample_label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_labels.append(int(parts[0]))
                    bboxes.append([float(x) for x in parts[1:5]])
    
    # Apply augmentations
    if bboxes:
        transform = get_augmentation_pipeline()
        
        fig, axes = plt.subplots(2, 4, figsize=(16, 8))
        axes = axes.flatten()
        
        # Original image
        axes[0].imshow(image)
        axes[0].set_title('Orijinal / Original')
        axes[0].axis('off')
        
        # Augmented images
        for i in range(1, 8):
            try:
                transformed = transform(image=image, bboxes=bboxes, class_labels=class_labels)
                aug_image = transformed['image']
                axes[i].imshow(aug_image)
                axes[i].set_title(f'Artırma {i} / Augmentation {i}')
                axes[i].axis('off')
            except:
                axes[i].imshow(image)
                axes[i].set_title(f'Artırma {i} (hata) / Aug {i} (error)')
                axes[i].axis('off')
        
        plt.suptitle('Veri Artırma Örnekleri / Data Augmentation Examples', fontsize=14)
        plt.tight_layout()
        plt.show()
else:
    print("No sample images found for augmentation demo.")

## Özet / Summary

Bu notebook şunları gösterdi:
- Veri seti yapısının analizi
- Sınıf dağılımının görselleştirilmesi
- Örnek görüntülerin yüklenmesi ve etiketlerin çizilmesi
- Bounding box boyutlarının analizi
- Veri artırma tekniklerinin görselleştirilmesi
---
This notebook demonstrated:
- Analyzing dataset structure
- Visualizing class distribution
- Loading sample images and drawing labels
- Analyzing bounding box sizes
- Visualizing data augmentation techniques